# How to Read Streaming Data From Socket in Databricks
* Topic: Introduction to How to Read Streaming Data From Socket in Databricks
* Author: Oindrila Chakraborty

# Objective
* 1. Read <b>Streaming</b> data from <b>Socket</b>, and, use the data in the <b>Streaming</b> application of counting the frequency of words coming as <b>Streaming</b> data.
* 2. Write the <b>Batch Code</b> first, and, then convert the <b>Batch Code</b> into <b>Streaming Code</b>.

# What is Socket?
* A <b>Socket</b> is basically an <b>end-point</b> in a <b>two-way communication</b>.
* In this current application, a <b>Socket</b> will be opened, and, some words, or, lines will be passed to it.
<br>The data coming from that <b>Socket</b> will be read by <b>Spark</b> as <b>Streaming</b> data for counting the number of times each of the words are coming.
  * <b>Socket</b> is very rarely used in <b>PRODUCTION</b> environment, but, is served as a very good tool for testing the <b>Streaming</b> applications.

# Strategy of Creating a Streaming Application That Reads Data from Socket
* The <b>Socket</b> will provide the data in form of lines.
* So, in order to get the words, the <b>text lines</b> need to be split into words.
* Suppose, the <b>Socket</b> sends the line <b>HELLO WORLD</b>.
  * The first task will be to break down this line into a <b>List</b>, containing the words <b>HELLO</b> and <b>WORLD</b>.
  <br>For this example, the line will be split with spaces (" ").
  * Once, the <b>List</b> containing the words is created, the <b>List</b> will be <b>exploded</b> to make sure all the values of this <b>List</b> will get into a particular column.
  <br>For this example, the created column will have two rows, i.e., <b>HELLO</b> and <b>WORLD</b> in each of the rows respectively.
  * Once, the column is created with the words from the line, this column can be used to <b>aggregate</b>, and, get the total count of each unique words, present in the column.
* The entire strategy for the <b>Streaming</b> application will be written in <b>Batch</b> format first.
* The the code written in the <b>Batch</b> format will be converted into <b>Streaming</b> format.

# Batch Format Code

#### Read Data from File
* To read data in the <b>Batch</b> format, the data is kept and read from a file.
<br>For this example, the data is kept in a <b>Text</b> file.
* Suppose, the <b>Text</b> file contains the below data -
  * <b>Soumya lives in Dhakuria East Road His home is east facing
* In the below code, a <b>DataFrame</b> is created to read data from the <b>Text</b> file.

In [0]:
# Create the DataFrame by Reading Text File
df_text_file = spark.read.format("text").load("/Workspace/Users/oindrila.chakraborty88@gmail.com/oindrila-streaming-data-repo/Files/socket_data.txt")
# Validate the Schema of the Created DataFrame
df_text_file.printSchema()
# Display the Content of the Created DataFrame
df_text_file.display()

* Once, the <b>DataFrame</b> is created and the <b>Schema</b> of that created <b>DataFrame</b> is validated, it can be seen that the <b>DataFrame</b> has only one column, i.e., <b>value</b> of <b>string</b> data type.
  * Since, no explicit <b>Schema</b> is defined for the created <b>DataFrame</b>, the name <b>value</b> is used by <b>Spark</b> for the name of the created column.
* This <b>value</b> column of the <b>DataFrame</b> will have the entire line that is present in the <b>Text</b> file.

#### Split the Line into Words
* Create a new <b>DataFrame</b> with a new column, i.e., <b>words</b> using <b>withColumn ()</b> function from the above created <b>DataFrame</b> object, i.e., <b>df_text_file</b>.
* The column <b>words</b> will contain all the words present in the column <b>value</b> of the <b>DataFrame</b> object, i.e., <b>df_text_file</b>.
<br>The <b>split ()</b> function will split the content of the column <b>value</b> of the <b>DataFrame</b> object, i.e., <b>df_text_file</b> using the separator <b>space</b>, i.e., " ", and, will generate the <b>List</b> of words, which will be stored in the column <b>words</b> in the new <b>DataFrame</b>.

In [0]:
from pyspark.sql.functions import split, lower

# Split the Line into Words
df_words = df_text_file.withColumn("words", split(lower("value"), " "))
# Display the Content of the Created DataFrame
df_words.display()

#### Explode the List of Words
* The <b>explode ()</b> will make sure that each of the elements of the <b>List</b> in the <b>words</b> column of the latest created <b>DataFrame</b>, i.e., <b>df_words</b>, comes into separate rows.
* Create a new <b>DataFrame</b> with a new column, i.e., <b>word</b> using <b>explode ()</b> function from the latest created <b>DataFrame</b> object, i.e., <b>df_words</b>, which will hold each word in each row.

In [0]:
from pyspark.sql.functions import explode

# Explode the List of Words
df_explode = df_words.withColumn("word", explode("words"))
# Display the Content of the Created DataFrame
df_explode.display()

* Since, the example needs only the column <b>word</b>, it is better to drop the other two columns, i.e., <b>value</b> and <b>words</b> using the <b>drop ()</b> function.
* Now, there will be only one column, and, each of the words will be in each of the rows.

In [0]:
# Drop Unnecessary Columns
df_explode = df_explode.drop("value", "words")
# Display the Content of the Created DataFrame
df_explode.display()

#### Aggregate the Words to Geneate Counts
* To write the aggregation, a new <b>DataFrame</b>, i.e., <b>df_agg</b> is created.
* Group by the column <b>word</b> of the latest created <b>DataFrame</b> object, i.e., <b>df_explode</b> and create a new column, i.e., <b>frequency_count</b> to hold the number of times each unique word is present in the column <b>word</b>, by using the <b>agg ()</b> function on the grouped data and passing the <b>count ()</b> function inside it.

In [0]:
from pyspark.sql.functions import lit, count

# Aggregate the Words to Geneate Counts
df_agg = df_explode.groupBy("word").agg(count(lit(1)).alias("frequency_count"))
# Display the Content of the Created DataFrame
df_agg.display()

# Change Code From Batch Format to Streaming Format

#### Read Data from Streaming Source
* To read data in the <b>Streaming</b> format, <b>spark.readStream ()</b> should be written, instead of <b>spark.read ()</b>.
* Since, the data needs to be read from the <b>Socket</b>, <b>format("socket")</b> should be written, instead of <b>format("text")</b>.
* Also, the <b>host name</b> and <b>port</b> details should be provided using the <b>option ()</b> function.
* Since, the data is to be read from <b>Socket</b> source, no parameter should be provided to the <b>load ()</b> function.

In [0]:
# Create the DataFrame by Reading Streaming Data From Socket
df_socket = spark.readStream().format("socket").option("host", "localhost").option("port", "9999").load()
# Validate the Schema of the Created DataFrame
df_socket.printSchema()

* There is no need to display the content of the <b>Streaming DataFrame</b>, because, the data will be read when the <b>Streaming</b> application is up.

#### Split the Line into Words
* The code will remain the same.
* The content of the created <b>DataFrame</b> will not be displayed as the <b>Streaming</b> application has not started.

In [0]:
from pyspark.sql.functions import split, lower

# Split the Line into Words
df_words = df_socket.withColumn("words", split(lower("value"), " "))
# Display the Content of the Created DataFrame
df_words.display()

#### Explode the List of Words
* The code will remain the same.
* The content of the created <b>DataFrame</b> will not be displayed as the <b>Streaming</b> application has not started.

In [0]:
from pyspark.sql.functions import explode

# Explode the List of Words
df_explode = df_words.withColumn("word", explode("words"))
# Display the Content of the Created DataFrame
df_explode.display()

In [0]:
# Drop Unnecessary Columns
df_explode = df_explode.drop("value", "words")
# Display the Content of the Created DataFrame
df_explode.display()

#### Aggregate the Words to Geneate Counts

* The code will remain the same.
* The content of the created <b>DataFrame</b> will not be displayed as the <b>Streaming</b> application has not started.

In [0]:
from pyspark.sql.functions import lit, count

# Aggregate the Words to Geneate Counts
df_agg = df_explode.groupBy("word").agg(count(lit(1)).alias("frequency_count"))
# Display the Content of the Created DataFrame
df_agg.display()

#### Write the Output to Console Streaming
* The aggregated <b>Streaming</b> data needs to be written to <b>Console</b>.
* <b>Console</b> is the terminal in which the output will be displayed.
* To write the <b>Streaming</b> output data, <b>spark.writeStream ()</b> should be written.
* Since, the output would be written to the <b>Console</b>, <b>format("console")</b> should be written, after <b>spark.writeStream ()</b>.
* An <b>Output Mode</b> shouuld be provided, where the output of the processed <b>Streaming</b> data will be displayed.
<br>For this example, provide <b>complete</b> as <b>Output Mode</b>.
* Then the <b>Streaming</b> application needs to be started by using the <b>start ()</b> function.
* Finally, the <b>awaitTermination ()</b> function is used to make sure that when the <b>Driver</b> of the <b>Cluster</b> is idle, and, the <b>Streaming</b> application is running on <b>Executor</b>s only, still then the <b>Driver</b> is connected to the <b>Executor</b>s. So, the <b>Driver</b> will not exit.

In [0]:
# Write the Output to Console Streaming
df_agg.writeStream().format("console").outputMode("complete").start()